# 13 · FlashAttention: online softmax and the IO model

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 10. Sources: Dao et al. (arXiv:2205.14135); Milakov & Gimelshein (arXiv:1805.02867).

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Show that tiled attention with an online softmax is exact, and count the HBM traffic it saves.

## Theory and equations
$m^{(j)} = \max(m^{(j-1)}, \operatorname{rowmax} S_j)$, $\ell^{(j)} = e^{m^{(j-1)}-m^{(j)}}\ell^{(j-1)} + \operatorname{rowsum} e^{S_j - m^{(j)}}$,
$\tilde o^{(j)} = e^{m^{(j-1)}-m^{(j)}}\tilde o^{(j-1)} + e^{S_j - m^{(j)}} V_j$, and $O = \tilde o/\ell$.
IO: naive $4Td + 4T^2$; tiled $\Theta(T^2d^2/M)$.

## Diagram
```
for each Q tile (in SRAM):            for each K,V tile (streamed):  S_tile -> update (m, l, o)
write O tile once                     the T x T matrix is never written to HBM
```

## Implementation

In [ ]:
from inference_lab import flash
import torch.nn.functional as F
torch.manual_seed(0)
q, k, v = (torch.randn(1, 4, 300, 64) for _ in range(3))      # (batch, heads, seq, head_dim)
for causal in (False, True):
    o, _ = flash.tiled_attention(q, k, v, 32, 32, causal=causal)
    ref = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
    print("causal" if causal else "full  ", "max |tiled - SDPA| =", float((o - ref).abs().max()))

## Experiment: HBM traffic (*theoretical*, FP16, M = 100k elements of SRAM)

In [ ]:
Ts = [512, 1024, 2048, 4096, 8192, 16384, 32768]
fig, ax = plt.subplots()
for d, a in [(64, .6), (128, 1)]:
    ax.plot(Ts, [2 * flash.io_naive(T, d) / 2**20 for T in Ts], "o-", color=C["waste"], alpha=a, label=f"naive d={d}")
    ax.plot(Ts, [2 * flash.io_flash(T, d, 100_000) / 2**20 for T in Ts], "o-", color=C["compute"], alpha=a, label=f"tiled d={d}")
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("T"); ax.set_ylabel("HBM traffic per head (MiB)"); ax.legend(); plt.show()
for d in (64, 128):
    print(f"d={d}: traffic ratio at T=16384 = {flash.io_naive(16384, d) / flash.io_flash(16384, d, 100_000):.1f}x  (M/2d^2 = {100_000/(2*d*d):.1f})")

## Results: extra memory

In [ ]:
for T in (4096, 32768, 131072):
    print(f"T={T:6d}: naive S+P per head = {2 * T * T * 2 / 2**30:8.2f} GiB   tiled extra = {T * 4 / 2**10:6.0f} KiB (LSE)")

## Interpretation
The result is exact. By this count the traffic saving is $\approx M/2d^2$: a few times at
$d=128$. The decisive change is memory, $O(T^2)$ to $O(T)$, and fusing three kernels into one.

## Limitations
PyTorch loops, not a fused kernel: no SRAM, warps or tensor cores. Real speedups depend on the kernel.

## Conclusion
FlashAttention is an IO and memory result. It does not change the size of the KV cache.